In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import random 
import pymysql

print("=" *80)
print(" 개인프로젝트 네이버 쇼핑 수집 (최종 완벽본)")
print("=" *80)
print("\n")

# -------------------------------------------------------------
# 1. 자동 로그인을 위한 네이버 ID/PW 및 파라미터 입력
# -------------------------------------------------------------
v_id = input('🔑 네이버 로그인 ID를 입력하세요: ')
v_passwd = input('🔑 네이버 로그인 비밀번호를 입력하세요: ')

query_txt = input('1. 검색할 네이버 쇼핑 키워드는 무엇입니까?: ').replace('"','')
cnt = int(input('2. 수집할 상품은 총 몇 건입니까?(예: 30): '))
# page_cnt는 네이버 쇼핑에서 쓰이지 않아 삭제했습니다.

f_dir = input("3. 파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\) [엔터시 기본경로 적용]:")
if f_dir=='' :
    f_dir='c:\\py_temp\\결과 추출 - 네이버쇼핑\\'

print("\n🚀 데이터 수집을 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!")

# 실행시간 측정 시작
s_time = time.time()

# 폴더 설정 로직
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)
os.makedirs(f_dir+s+'-'+query_txt, exist_ok=True)
os.chdir(f_dir+s+'-'+query_txt)
ff_name = f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name = f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name = f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xls'

# -------------------------------------------------------------
# 2. 크롬 드라이버 셋팅
# -------------------------------------------------------------
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options) 
driver.maximize_window()

try:
    # -------------------------------------------------------------
    # 3. 네이버 자동 로그인 (자바스크립트 주입 기법이 가장 안전합니다!)
    # -------------------------------------------------------------
    driver.get("https://nid.naver.com/nidlogin.login")
    time.sleep(2)

    driver.execute_script(f"document.getElementsByName('id')[0].value='{v_id}'")
    driver.execute_script(f"document.getElementsByName('pw')[0].value='{v_passwd}'")
    time.sleep(1)
    
    driver.find_element(By.XPATH,'//*[@id="log.login"]').click()  
    print(">> 성공적으로 네이버 로그인 시도 중...")
    time.sleep(3) 

    # -------------------------------------------------------------
    # 4. 로그인 완료 후 네이버 쇼핑으로 이동!
    # -------------------------------------------------------------
    driver.get("https://shopping.naver.com/ns/home")
    time.sleep(3) 
    
    wait = WebDriverWait(driver, 10)
    search_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[placeholder="상품명 또는 브랜드 입력"]')))
    
    search_input.click()
    search_input.clear()
    
    # 여기서는 검색어이므로 봇 탐지를 피하기 위해 사람처럼 한 글자씩 칩니다!
    for a in query_txt:
        search_input.send_keys(a)
        time.sleep(random.uniform(0.1, 0.35))
        
    time.sleep(2) 
    search_input.send_keys(Keys.ENTER) 
    
    print(f">> 네이버 쇼핑 '{query_txt}' 단어 타자 입력 & 검색 완료! 결과창 렌더링 대기...")
    time.sleep(3) 

except Exception as e:
    print("\n[접속 에러 발생] 진행 도중 문제가 생겼습니다:", e)


# ==========================================================
# 5. 크롤링 영역 (find 함수 사용 버전 기준)
# ==========================================================
product_names = []   
prices = []          
discounts = []       
stars = []           
review_cnts = []     
total_count = 0  

print('\n상품 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~')
time.sleep(3) 

seen_products = set()

while total_count < cnt:
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    
    # 바둑판형(Grid) 상품들을 감싸고 있는 전체 li를 찾습니다.
    # 선생님이 주신 HTML에서 'composite_card_container' 클래스는 고정되어 변하지 않는 이름표입니다.
    items = soup.find_all('li', class_='composite_card_container')
    
    if len(items) == 0:
        print("🚨 화면에서 상품을 찾지 못했습니다. 클래스명이 맞는지 다시 확인해주세요!")
    
    start_count = total_count 
        
    for item in items:
        if total_count >= cnt:
            break
            
        # 1. 제품 이름
        try:
            name_node = item.find('strong', class_='productCardTitle_product_card_title__eQupA')
            if name_node:
                name = name_node.get_text(strip=True)
            else:
                name = "이름 없음"
        except:
            name = "이름 없음"

        # 중복 체크 (수집했던 상품이면 패스)
        if name in seen_products or name == "이름 없음":
            continue
            
        seen_products.add(name)
        total_count += 1
        
        print(f"🚀 총 {cnt}건 중 {total_count}번째 상품 수집 중 =========")
        f = open(ff_name, 'a', encoding='UTF-8')
        f.write("\n")
        f.write(f"[{total_count} 번째 상품 정보]====\n")
        f.write("1.제품의 이름: " + name + "\n")
        product_names.append(name)
        
        # 2. 가격
        try:
            price_node = item.find('span', class_='priceTag_price__hGtfm')
            if price_node:
                price = price_node.get_text(strip=True)
            else:
                price = "0"
        except:
            price = "0"
            
        # 선생님 요청대로 뒤에 "원"을 붙여서 기록합니다.
        f.write("2.제품 판매가: " + price + "원\n")
        prices.append(price)

        # 3. 할인율
        try:
            dc_node = item.find('span', class_='priceTag_discount_ratio__VE866')
            if dc_node:
                # "31% 할인" -> "31" 로 숫자만 깔끔하게 남깁니다.
                discount = dc_node.get_text(strip=True).replace("할인", "").replace("%", "")
            else:
                discount = "0"
        except:
            discount = "0"
            
        f.write("3.할인율: " + discount + "%\n")
        discounts.append(discount)

        # 4. 리뷰 별점
        try:
            star_node = item.find('span', class_='productCardReview_star__7iHNO')
            if star_node:
                # "별점4.74" -> "4.74" 로 변경
                star = star_node.get_text(strip=True).replace("별점", "")
            else:
                star = "0"
        except:
            star = "0"
            
        f.write("4.리뷰 별점: " + star + "\n")
        stars.append(star)

        # 5. 리뷰 개수
        try:
            # "productCardReview_text__A9N9N" 클래스를 가진 span이 별점에도 있어서 여러 개일 수 있습니다.
            # 전부 찾아서 반복문으로 "리뷰" 글자가 포함된 태그만 쏙 빼옵니다!
            review_nodes = item.find_all('span', class_='productCardReview_text__A9N9N')
            review_cnt = "0"
            for r_node in review_nodes:
                r_text = r_node.get_text(strip=True)
                if "리뷰" in r_text:
                    # "리뷰 10,611" -> "10,611"
                    review_cnt = r_text.replace("리뷰", "").strip()
                    break
        except:
            review_cnt = "0"
            
        f.write("5.리뷰 개수: " + review_cnt + "\n")
        review_cnts.append(review_cnt)

        f.close()
        time.sleep(0.1)

    if total_count >= cnt:
        print(f"\n✅ 수집 목표량({cnt}건) 달성 완료!")
        break
    else:
        # 새로 추가할 상품 로딩을 위한 사람다운 페이지 다운 스크롤
        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
        time.sleep(1)
        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
        time.sleep(2)


# ==========================================================
# 6. Step 7. xls 형태와 csv 형태로 저장하기
# ==========================================================
import pandas as pd
news_reple = pd.DataFrame()
news_reple['제품의 이름'] = pd.Series(product_names)
news_reple['제품 판매가'] = pd.Series(prices)
news_reple['할인율'] = pd.Series(discounts)
news_reple['리뷰 별점'] = pd.Series(stars)
news_reple['리뷰 개수'] = pd.Series(review_cnts)

news_reple.to_csv(fc_name, encoding="utf-8-sig", index=False)
news_reple.to_excel(fx_name, index=False, engine='openpyxl')


# ==========================================================
# 7. Step 8. 수집한 5가지 정보를 통째로 MySQL DB에 넣기!
# ==========================================================
try:
    conn = pymysql.connect(
        host='localhost',         
        user='root',              
        password='Jx03151616~~',  
        db='youtube_db',  
        charset='utf8mb4',        
        cursorclass=pymysql.cursors.DictCursor
    )
    with conn.cursor() as cursor:
        # DB 구조도 선생님이 원하신 5개 속성에 딱 맞게 생성합니다!
        create_table_sql = """
        CREATE TABLE IF NOT EXISTS naver_shopping_products (
            id INT AUTO_INCREMENT PRIMARY KEY, 
            product_name VARCHAR(255),         
            price VARCHAR(50),                 
            discount VARCHAR(50),
            star VARCHAR(50),
            review_cnt VARCHAR(50)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        cursor.execute(create_table_sql)
        
        insert_sql = """
        INSERT INTO naver_shopping_products (product_name, price, discount, star, review_cnt) 
        VALUES (%s, %s, %s, %s, %s)
        """
        # 수집한 상품 개수만큼 반복하며 DB에 한 줄씩 기록!
        for i in range(len(product_names)):
            cursor.execute(insert_sql, (
                product_names[i], 
                prices[i], 
                discounts[i],
                stars[i],
                review_cnts[i]
            ))
        conn.commit()
        print(f"🎉 짝짝짝! 네이버 쇼핑 DB 저장까지 완벽하게 완료되었습니다!")
        
except Exception as e:
    print(f"\n[DB 에러] DB 저장 중 에러가 발생했습니다: {e}")
finally:
    try:
        conn.close()
    except:
        pass


# ==========================================================
# 8. Step 9. 요약 정보 출력하기
# ==========================================================
e_time = time.time( )
t_time = e_time - s_time

print("\n")
print("=" *120)
print(f"1.모든 작업 종료. 수집된 전체 상품 수는 {total_count} 건 입니다.")
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *120)

driver.quit() # 안전하게 창 닫기


 개인프로젝트 네이버 쇼핑 수집 (최종 완벽본)



🚀 데이터 수집을 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!
>> 성공적으로 네이버 로그인 시도 중...
>> 네이버 쇼핑 '런닝화' 단어 타자 입력 & 검색 완료! 결과창 렌더링 대기...

상품 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~
🚀 총 30건 중 1번째 상품 수집 중 =========
🚀 총 30건 중 2번째 상품 수집 중 =========
🚀 총 30건 중 3번째 상품 수집 중 =========
🚀 총 30건 중 4번째 상품 수집 중 =========
🚀 총 30건 중 5번째 상품 수집 중 =========
🚀 총 30건 중 6번째 상품 수집 중 =========
🚀 총 30건 중 7번째 상품 수집 중 =========
🚀 총 30건 중 8번째 상품 수집 중 =========
🚀 총 30건 중 9번째 상품 수집 중 =========
🚀 총 30건 중 10번째 상품 수집 중 =========
🚀 총 30건 중 11번째 상품 수집 중 =========
🚀 총 30건 중 12번째 상품 수집 중 =========
🚀 총 30건 중 13번째 상품 수집 중 =========
🚀 총 30건 중 14번째 상품 수집 중 =========
🚀 총 30건 중 15번째 상품 수집 중 =========
🚀 총 30건 중 16번째 상품 수집 중 =========
🚀 총 30건 중 17번째 상품 수집 중 =========
🚀 총 30건 중 18번째 상품 수집 중 =========
🚀 총 30건 중 19번째 상품 수집 중 =========
🚀 총 30건 중 20번째 상품 수집 중 =========
🚀 총 30건 중 21번째 상품 수집 중 =========
🚀 총 30건 중 22번째 상품 수집 중 =========
🚀 총 30건 중 23번째 상품 수집 중 =========
🚀 총 30건 중 24번째 상품 수집 중 =========
🚀 총 30건 중 25번째 상품 수집 중 =========
🚀 총 30건 